SQS and SNS are AWS's core messaging services for building decoupled, event-driven architectures. SQS is a fully managed message queue — producers send messages, consumers poll and process them independently. SNS is a pub/sub notification service — publishers send to a topic and all subscribers receive a copy simultaneously. Together they are the foundation of scalable, loosely coupled systems on AWS and appear extensively on the SAA-C03 exam.

## Why Decouple with Messaging?

```text
Tightly coupled:                   Loosely coupled (SQS):
Frontend → Backend (sync)          Frontend → SQS Queue → Backend
  If backend is slow/down,           Frontend always succeeds
  frontend fails or waits             Backend processes at its own pace
```

Decoupling with a queue means:
- **Producers** and **consumers** scale independently
- Temporary backend failures don't drop messages — they stay in the queue
- Traffic spikes are absorbed by the queue instead of overwhelming the backend
- Classic exam pattern: EC2 frontend puts jobs in SQS → backend worker fleet scales with queue depth via Auto Scaling

## SQS Fundamentals

### How SQS works
1. **Producer** calls `SendMessage` → message stored in queue
2. **Consumer** calls `ReceiveMessage` → SQS delivers up to 10 messages
3. Consumer processes the message, then calls `DeleteMessage` to remove it
4. If `DeleteMessage` is never called, the message reappears in the queue after the **visibility timeout** expires

```text
Producer ──SendMessage──▶ [ SQS Queue ] ──ReceiveMessage──▶ Consumer
                                │                              │
                          message hidden                  DeleteMessage
                          (visibility timeout)            (on success)
                                │
                          reappears if not deleted
                          (consumer crashed → retry)
```

### Key limits and settings

| Setting | Default | Range / Notes |
|---|---|---|
| **Message size** | — | Up to 256 KB; use S3 + pointer for larger payloads |
| **Message retention** | 4 days | 1 minute – 14 days |
| **Visibility timeout** | 30 seconds | 0 – 12 hours; should exceed max processing time |
| **Delivery delay** | 0 | 0 – 15 minutes (Delay Queue) |
| **Receive wait time** | 0 seconds | 0 – 20 seconds (Long Polling) |
| **In-flight messages** | — | Standard: 120,000; FIFO: 20,000 |
| **Batch size** | — | 1 – 10 messages per ReceiveMessage call |

## Standard vs FIFO Queues

| | Standard Queue | FIFO Queue |
|---|---|---|
| **Ordering** | Best-effort (not guaranteed) | **Strict FIFO** within a message group |
| **Delivery** | At-least-once (duplicates possible) | **Exactly-once** processing |
| **Throughput** | Nearly unlimited | 300 msg/s (3,000 with batching) |
| **Use case** | High-throughput, order doesn't matter | Order-critical: payments, inventory, commands |
| **Naming** | Any name | Must end in `.fifo` |

### FIFO concepts
- **Message Group ID**: messages with the same group ID are processed in strict order. Different group IDs are processed independently in parallel — this is how FIFO queues scale.
- **Message Deduplication ID**: SQS deduplicates messages with the same ID within a 5-minute window — prevents exactly-once delivery failures from retries.
- **Content-based deduplication**: SQS auto-generates the deduplication ID from an SHA-256 hash of the message body.

> **Exam tip:** Order + exactly-once → FIFO. High throughput, order irrelevant → Standard.

## Visibility Timeout and Dead-Letter Queues

### Visibility timeout
When a consumer receives a message, SQS makes it invisible to other consumers for the **visibility timeout** duration. This prevents two consumers from processing the same message simultaneously.

- If the consumer processes and deletes the message before the timeout → success
- If the consumer crashes or takes too long → timeout expires → message becomes visible again → another consumer picks it up
- **Set visibility timeout longer than your maximum processing time** — if it's too short, messages get processed twice
- Consumers can call `ChangeMessageVisibility` to extend the timeout if processing takes longer than expected

### Dead-Letter Queue (DLQ)
A DLQ receives messages that fail processing too many times.

```text
Main Queue ──(fails N times)──▶ Dead-Letter Queue
                                      │
                               inspect / debug / replay
```

- Configure **maxReceiveCount** on the source queue (e.g., 3 — move to DLQ after 3 failed attempts)
- The DLQ must be the **same type** as the source queue (Standard DLQ for Standard; FIFO DLQ for FIFO)
- DLQs are used to isolate and debug poison messages without blocking the main queue
- **DLQ Redrive**: replay messages from the DLQ back to the source queue after fixing the bug

## Long Polling vs Short Polling

### Short polling (default)
- `ReceiveMessage` returns immediately — even if the queue is empty
- Results in many empty responses and unnecessary API calls when the queue is idle
- You pay for every API call

### Long polling
- Set `WaitTimeSeconds` (1–20 seconds) on `ReceiveMessage`
- SQS waits up to that duration for a message to arrive before returning
- Reduces empty responses, lowers costs, reduces CPU spin on consumer
- **Always prefer long polling** — there is no downside for most workloads

### Delay Queue
Set a **Delivery Delay** (0–15 minutes) on the queue or per-message. New messages are invisible to consumers until the delay expires. Use case: defer processing (e.g., send a follow-up email 10 minutes after sign-up).

> **Exam tip:** "Reduce empty receives" or "reduce polling costs" → enable Long Polling.

## SQS Extended Client and Security

### Large messages
SQS messages are limited to 256 KB. For larger payloads:
- Store the payload in **S3**
- Put a **pointer (S3 reference)** in the SQS message
- Consumer reads the pointer, fetches the payload from S3
- The **SQS Extended Client Library** (Java/Python) handles this automatically

### Security
- **Encryption at rest**: SQS Server-Side Encryption (SSE) using KMS keys
- **Encryption in transit**: HTTPS endpoints (enforced by queue policy)
- **Access control**:
  - **IAM policies**: control which principals can call SQS API actions
  - **SQS Queue Policy** (resource-based): control cross-account access or allow SNS/S3 to publish to the queue
- **VPC Endpoints**: access SQS from within a VPC without traversing the public internet

## SNS Fundamentals

SNS is a fully managed **pub/sub** (publish/subscribe) messaging service.

### How SNS works
1. Create an **SNS Topic**
2. **Subscribers** register to receive messages from the topic
3. **Publisher** sends one message to the topic → SNS **fans out** to all subscribers simultaneously

```text
                     ┌──▶ SQS Queue (async processing)
Publisher ──▶ SNS ───┼──▶ Lambda (event processing)
   Topic             ├──▶ HTTP/HTTPS endpoint (webhook)
                     ├──▶ Email / SMS
                     └──▶ Mobile Push (APNS, FCM)
```

### Supported subscription protocols
- SQS
- Lambda
- HTTP / HTTPS
- Email / Email-JSON
- SMS
- Mobile push (Apple APNS, Google FCM, Amazon ADM)
- Kinesis Data Firehose (deliver notifications directly to S3/Redshift/OpenSearch)

### Key limits
- Up to **12.5 million subscriptions** per topic
- Up to **100,000 topics** per account
- Message size: up to **256 KB**

## SNS Standard vs FIFO Topics

| | SNS Standard | SNS FIFO |
|---|---|---|
| **Ordering** | Best-effort | Strict order per Message Group ID |
| **Delivery** | At-least-once | Exactly-once |
| **Subscribers** | SQS, Lambda, HTTP, Email, SMS, Mobile | **SQS FIFO only** |
| **Throughput** | Very high | 300 msg/s (3,000 with batching) |
| **Use case** | Fanout to many subscriber types | Ordered, deduplicated fan-out |

> **Exam tip:** SNS FIFO can only deliver to **SQS FIFO queues**. If you need FIFO ordering end-to-end, use SNS FIFO → SQS FIFO.

## SNS Message Filtering

By default, every subscriber receives every message. **Subscription filter policies** let each subscriber receive only the messages it cares about.

```text
Publisher sends:  { "eventType": "ORDER_PLACED", "status": "new" }

Subscription A filter: { "eventType": ["ORDER_PLACED"] }  → receives it
Subscription B filter: { "eventType": ["ORDER_SHIPPED"] } → does NOT receive it
Subscription C: no filter                                 → receives all messages
```

Filter policies match on **message attributes** (or message body with body-based filtering). Conditions support exact match, prefix matching, numeric ranges, and existence checks.

Use case: a single order-events topic serving multiple microservices — each subscribes with a filter for only its relevant event types. No need for separate topics per event type.

## SNS + SQS Fan-Out Pattern

The most important architecture pattern combining SNS and SQS:

```text
S3 Event / Application
        │
        ▼
    SNS Topic
    /    |    \
SQS-A  SQS-B  SQS-C
  │      │      │
Worker  Worker  Worker
(email) (ship)  (analytics)
```

**Why not publish directly to multiple SQS queues?**
- Application would need to call `SendMessage` N times — tightly coupled, misses queues if one fails
- SNS delivers atomically to all subscribers; producers write once

**Fan-out benefits:**
- Fully decoupled — each worker subscribes independently; adding a new consumer requires no producer change
- Each SQS queue can have its own DLQ, visibility timeout, and scaling policy
- Data persistence: messages stay in SQS queues even if the worker is down

### S3 event fan-out
S3 supports only one event notification destination per event type. To fan out S3 events to multiple consumers: S3 → SNS → multiple SQS queues.

> **Exam tip:** Whenever you need one event delivered to multiple downstream systems reliably → SNS + SQS fan-out.

## SQS vs SNS vs EventBridge

| | SQS | SNS | EventBridge |
|---|---|---|---|
| Model | Queue (pull) | Pub/Sub (push) | Event bus (push) |
| Consumers | One (or competing) | Many simultaneously | Many with rules |
| Persistence | Yes — messages queued | No — fire and forget | No |
| Filtering | No (DLQ only) | Subscription filter policies | Rich rule-based filtering |
| Sources | Application push | Application push | AWS services + SaaS + custom |
| Use case | Async work queue, buffer | Fan-out notifications | Event routing across services |

> **Exam pattern:**
> - Decouple producer/consumer + buffer traffic → **SQS**
> - One event → many subscribers simultaneously → **SNS**
> - Route AWS service events (CloudWatch, S3, EC2) with complex rules → **EventBridge**

## Working with SQS and SNS via boto3

In [ ]:
import boto3
import json

sqs = boto3.client('sqs', region_name='us-east-1')
sns = boto3.client('sns', region_name='us-east-1')

In [ ]:
# Create a Standard SQS queue with DLQ

# 1. Create the DLQ first
dlq = sqs.create_queue(
    QueueName='orders-dlq',
    Attributes={
        'MessageRetentionPeriod': '1209600',  # 14 days
    }
)
dlq_url = dlq['QueueUrl']
dlq_arn  = sqs.get_queue_attributes(QueueUrl=dlq_url,
                                     AttributeNames=['QueueArn'])['Attributes']['QueueArn']

# 2. Create the main queue pointing to the DLQ
queue = sqs.create_queue(
    QueueName='orders',
    Attributes={
        'VisibilityTimeout':       '60',        # 60s — consumer must delete within this
        'MessageRetentionPeriod':  '345600',    # 4 days
        'ReceiveMessageWaitTimeSeconds': '20',  # long polling
        'RedrivePolicy': json.dumps({
            'deadLetterTargetArn': dlq_arn,
            'maxReceiveCount':     3            # DLQ after 3 failed attempts
        }),
        'SqsManagedSseEnabled': 'true'          # SSE encryption
    }
)
queue_url = queue['QueueUrl']
print(f"Queue created: {queue_url}")

In [ ]:
# Send a message
sqs.send_message(
    QueueUrl=queue_url,
    MessageBody=json.dumps({'orderId': 'ORD-001', 'amount': 99.99}),
    MessageAttributes={
        'eventType': {'DataType': 'String', 'StringValue': 'ORDER_PLACED'}
    }
)

# Receive and process
response = sqs.receive_message(
    QueueUrl=queue_url,
    MaxNumberOfMessages=10,
    WaitTimeSeconds=20,          # long polling
    MessageAttributeNames=['All']
)

for message in response.get('Messages', []):
    body = json.loads(message['Body'])
    print(f"Processing order: {body['orderId']}")

    # --- do processing here ---

    # Delete on success
    sqs.delete_message(
        QueueUrl=queue_url,
        ReceiptHandle=message['ReceiptHandle']
    )
    print("Message deleted")

In [ ]:
# Create a FIFO queue
fifo_queue = sqs.create_queue(
    QueueName='payments.fifo',      # name MUST end in .fifo
    Attributes={
        'FifoQueue':                    'true',
        'ContentBasedDeduplication':    'true',   # deduplicate by body hash
        'VisibilityTimeout':            '60',
        'ReceiveMessageWaitTimeSeconds': '20',
    }
)
fifo_url = fifo_queue['QueueUrl']

# Send an ordered message
sqs.send_message(
    QueueUrl=fifo_url,
    MessageBody=json.dumps({'paymentId': 'PAY-001', 'amount': 250.00}),
    MessageGroupId='customer-42',    # all messages for same customer stay ordered
)
print("FIFO message sent")

In [ ]:
# Create SNS topic and subscribe SQS queues (fan-out pattern)
topic = sns.create_topic(Name='order-events')
topic_arn = topic['TopicArn']

# Subscribe the orders SQS queue
queue_arn = sqs.get_queue_attributes(
    QueueUrl=queue_url, AttributeNames=['QueueArn']
)['Attributes']['QueueArn']

subscription = sns.subscribe(
    TopicArn=topic_arn,
    Protocol='sqs',
    Endpoint=queue_arn,
    Attributes={
        'FilterPolicy': json.dumps({
            'eventType': ['ORDER_PLACED', 'ORDER_CANCELLED']
        })
    }
)
print(f"SQS subscribed with filter: {subscription['SubscriptionArn']}")

# Allow SNS to publish to the SQS queue (queue resource policy required)
# (In practice, set via sqs.set_queue_attributes with a Policy allowing SNS ARN)

In [ ]:
# Publish a message to SNS — fans out to all matching subscribers
sns.publish(
    TopicArn=topic_arn,
    Message=json.dumps({'orderId': 'ORD-002', 'amount': 149.00}),
    Subject='New Order',
    MessageAttributes={
        'eventType': {'DataType': 'String', 'StringValue': 'ORDER_PLACED'}
    }
)
print("SNS message published to all subscribers")

## Summary

| Concept | Key Takeaway |
|---|---|
| SQS | Pull-based queue; producers and consumers decoupled |
| Standard queue | Best-effort ordering; at-least-once; unlimited throughput |
| FIFO queue | Strict order per group; exactly-once; 300 msg/s (3,000 batched); name ends `.fifo` |
| Visibility timeout | Hides message during processing; set longer than max processing time |
| Dead-Letter Queue | Receives messages after maxReceiveCount failures; same type as source |
| Long polling | WaitTimeSeconds 1–20; reduces empty responses and cost |
| Delay Queue | Delivery delay 0–15 min; message invisible until delay expires |
| Large messages | Store in S3, put pointer in SQS; Extended Client Library handles automatically |
| SQS security | KMS SSE at rest; HTTPS in transit; IAM + queue policy for access control |
| SNS | Push-based pub/sub; one publish → all subscribers receive simultaneously |
| SNS subscribers | SQS, Lambda, HTTP/HTTPS, Email, SMS, Mobile Push, Kinesis Firehose |
| SNS FIFO | Ordered + exactly-once fan-out; only SQS FIFO subscribers |
| SNS filter policy | Each subscriber receives only matching messages; reduces noise |
| SNS + SQS fan-out | One event → SNS → multiple SQS queues → independent workers |
| S3 fan-out | S3 → SNS → multiple SQS queues (S3 only supports one destination per event) |
| SQS vs SNS | SQS: queue + buffer; SNS: simultaneous fan-out |
| vs EventBridge | EventBridge: rich rule-based routing of AWS/SaaS events |